# Controlled Descent Simulator — Model Derivation

**Author:** Diego Perazzolo, 2026
**License:** MIT

This notebook derives the full pipeline for the 6DOF rocket controlled descent simulator:

1. **Symbolic 6DOF model** (Newton–Euler in body/world frames)
2. **Linearization** at hover and **augmented system** with position integrators
3. **LQR** controller design on the augmented dynamics
4. **Feedforward via differential flatness** using `atan2` for full 4-quadrant coverage
5. **Reference trajectory** as degree-4 polynomials with smooth-landing boundary conditions
6. **Closed-loop simulation** with FF + LQR + anti-windup
7. **Realistic test** with aerodynamic drag active and a small initial perturbation
8. **Discussion of limitations** of the simplified approach
9. **C code export** of feedforward, dynamics and gain matrix for the WASM runtime

The runtime (C++/WASM) lives in the 'core' folder


In [ ]:
import sympy as sp
from sympy import (
    symbols, Function, Matrix, eye, zeros,
    sin, cos, sqrt, atan2, diff, simplify, Rational, solve, lambdify
)
sp.init_printing(use_latex='mathjax', wrap_line=False)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import control as ct

print("SymPy:", sp.__version__)
print("NumPy:", np.__version__)
print("control:", ct.__version__)

## 1. Symbolic 6DOF model

The rocket is a rigid body with mass `m` and diagonal inertia tensor `diag(Ix, Iy, Iz)`.
Its body frame has the thrust axis aligned with the body z-axis.
Three Euler angles `α` (pitch), `β` (yaw), `ψ` (roll) parameterize the attitude. Note: for a rocket, the body z-axis is the thrust/longitudinal axis, so a rotation around it (`ψ`) is what we conventionally call **roll** (the rocket spinning around itself), while a rotation around the body x-axis (`β`) is **yaw** (a lateral tilt). We keep the original symbol names (α, β, ψ) for backward compatibility with the earlier Maple derivation.

The rotation from body to world is built as **Rm = Rf1·Rf2·Rf3**:

- `Rf1(α)`: rotation of α around Y
- `Rf2(β)`: rotation of β around X
- `Rf3(ψ)`: rotation of ψ around Z



In [ ]:
# ==== Time and physical parameters ====
t = symbols('t', real=True)
m, Ix, Iy, Iz, g, F1_max = symbols('m I_x I_y I_z g F1_max', positive=True, real=True)
c, cz = symbols('c c_z', positive=True, real=True)   # drag coefficients

# ==== State variables (functions of t) ====
x     = Function('x')(t)
y     = Function('y')(t)
z     = Function('z')(t)
alpha = Function(r'\alpha')(t)
beta  = Function(r'\beta')(t)
psi   = Function(r'\psi')(t)

# ==== Control inputs ====
F1 = Function('F_1')(t)   # thrust along body z
T1 = Function('T_1')(t)   # torque around body x
T2 = Function('T_2')(t)   # torque around body y
T3 = Function('T_3')(t)   # torque around body z

In [ ]:
# ==== Reference frames ====
Rf1 = Matrix([                    # rotation of alpha around Y
    [ cos(alpha), 0, sin(alpha)],
    [ 0,          1, 0          ],
    [-sin(alpha), 0, cos(alpha) ]
])
Rf2 = Matrix([                    # rotation of beta around X
    [1, 0,          0          ],
    [0, cos(beta), -sin(beta)  ],
    [0, sin(beta),  cos(beta)  ]
])
Rf3 = Matrix([                    # rotation of psi around Z
    [cos(psi), -sin(psi), 0],
    [sin(psi),  cos(psi), 0],
    [0,         0,        1]
])

# Composite body -> world rotation
Rm = simplify(Rf1 * Rf2 * Rf3)
Rm

### Visualizing Rm: rocket body frame vs. world frame

The world frame `(X, Y, Z)` is fixed (Z up, gravity along `-Z`).
The body frame `(x_b, y_b, z_b)` is attached to the rocket; thrust acts along `+z_b`.
The rotation `Rm` maps a vector expressed in body coordinates to world coordinates:
`v_world = Rm · v_body`.

The figure below shows a rocket tilted by the three Euler angles.


In [ ]:
from IPython.display import SVG, display

# Static SVG illustrating the body and world frames during a tilted descent.
# Pitch alpha around Y is the dominant tilt shown here for clarity.
svg_rm = """
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 360" width="520" height="360"
     font-family="ui-sans-serif, system-ui, sans-serif" font-size="13">
  <defs>
    <marker id="arr" viewBox="0 0 10 10" refX="9" refY="5"
            markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M0,0 L10,5 L0,10 z" fill="#222"/>
    </marker>
    <marker id="arrW" viewBox="0 0 10 10" refX="9" refY="5"
            markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M0,0 L10,5 L0,10 z" fill="#1f77b4"/>
    </marker>
    <marker id="arrB" viewBox="0 0 10 10" refX="9" refY="5"
            markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M0,0 L10,5 L0,10 z" fill="#d62728"/>
    </marker>
  </defs>

  <!-- ground -->
  <line x1="20" y1="320" x2="500" y2="320" stroke="#888" stroke-width="1"/>
  <text x="460" y="338" fill="#666">ground</text>

  <!-- World frame at origin (lower-left) -->
  <g transform="translate(80,290)">
    <!-- X world: right -->
    <line x1="0" y1="0" x2="80" y2="0" stroke="#1f77b4" stroke-width="2" marker-end="url(#arrW)"/>
    <text x="86" y="5" fill="#1f77b4">X</text>
    <!-- Y world: into the page, drawn as diagonal back-left for perspective -->
    <line x1="0" y1="0" x2="-40" y2="20" stroke="#1f77b4" stroke-width="2" marker-end="url(#arrW)"/>
    <text x="-58" y="28" fill="#1f77b4">Y</text>
    <!-- Z world: up -->
    <line x1="0" y1="0" x2="0" y2="-90" stroke="#1f77b4" stroke-width="2" marker-end="url(#arrW)"/>
    <text x="6" y="-92" fill="#1f77b4">Z</text>
    <circle cx="0" cy="0" r="3" fill="#1f77b4"/>
    <text x="-22" y="18" fill="#1f77b4">O</text>
  </g>

  <!-- Rocket body, tilted by alpha around Y (pitch).
       Body z_b axis = direction of thrust. We tilt the rocket ~25 deg from vertical. -->
  <g transform="translate(310,200) rotate(-25)">
    <!-- rocket body (rectangle + cone) -->
    <rect x="-14" y="-50" width="28" height="100" fill="#eee" stroke="#333" stroke-width="1.5"/>
    <polygon points="-14,-50 14,-50 0,-78" fill="#ddd" stroke="#333" stroke-width="1.5"/>
    <!-- fins -->
    <polygon points="-14,50 -28,62 -14,62" fill="#bbb" stroke="#333" stroke-width="1"/>
    <polygon points="14,50 28,62 14,62" fill="#bbb" stroke="#333" stroke-width="1"/>
    <!-- thrust plume -->
    <polygon points="-9,62 9,62 0,90" fill="#ff9933" opacity="0.7"/>
    <polygon points="-5,62 5,62 0,80" fill="#ffe066" opacity="0.9"/>

    <!-- Body frame at the rocket centre of mass -->
    <!-- z_b: pointing up along the rocket axis (in the rotated frame, that's straight up) -->
    <line x1="0" y1="0" x2="0" y2="-70" stroke="#d62728" stroke-width="2" marker-end="url(#arrB)"/>
    <text x="6" y="-74" fill="#d62728">z_b</text>
    <!-- x_b: perpendicular to body axis, to the right -->
    <line x1="0" y1="0" x2="55" y2="0" stroke="#d62728" stroke-width="2" marker-end="url(#arrB)"/>
    <text x="60" y="5" fill="#d62728">x_b</text>
    <!-- y_b: into the page (diagonal) -->
    <line x1="0" y1="0" x2="-30" y2="18" stroke="#d62728" stroke-width="2" marker-end="url(#arrB)"/>
    <text x="-46" y="24" fill="#d62728">y_b</text>
    <circle cx="0" cy="0" r="3" fill="#d62728"/>
  </g>

  <!-- Tilt angle alpha drawn as an arc near the rocket -->
  <g transform="translate(310,200)">
    <!-- vertical reference -->
    <line x1="0" y1="0" x2="0" y2="-70" stroke="#888" stroke-width="1" stroke-dasharray="4 3"/>
    <!-- arc from vertical to z_b (tilted -25 deg) -->
    <path d="M 0,-50 A 50,50 0 0,0 -21.13,-45.32" fill="none" stroke="#222" stroke-width="1.2"/>
    <text x="-30" y="-58" fill="#222" font-style="italic">α</text>
  </g>

  <!-- Title and legend -->
  <text x="20" y="28" font-weight="bold" font-size="15">Body frame vs. world frame</text>
  <text x="20" y="48" fill="#1f77b4">— world axes (X, Y, Z): inertial, Z up</text>
  <text x="20" y="66" fill="#d62728">— body axes (x_b, y_b, z_b): attached to the rocket; thrust along +z_b</text>
  <text x="20" y="84" fill="#222">α, β, ψ: pitch, yaw, roll  →  Rm = Rf1(α)·Rf2(β)·Rf3(ψ)</text>
  <text x="20" y="102" fill="#222">v_world = Rm · v_body</text>
</svg>
"""
display(SVG(svg_rm))

### Newton–Euler equations

In world frame:

`m · a_world = F_thrust_world + F_gravity + F_drag_world`

with thrust along body `z`, gravity along `-Z` world, and drag proportional to **body** velocity:

`F_drag_body = -diag(c, c, c_z) · v_body`,    `v_body = Rm.T · v_world`,    `F_drag_world = Rm · F_drag_body`

Rotational dynamics use the diagonal-inertia approximation (the `Ix·α̈, Iy·β̈, Iz·ψ̈ = T_i` form).


In [ ]:
# ==== Newton–Euler ====
F_thrust_world = Rm * Matrix([0, 0, F1])

vel_world  = Matrix([diff(x, t), diff(y, t), diff(z, t)])
vel_body   = Rm.T * vel_world
drag_body  = -Matrix([c, c, cz]).multiply_elementwise(vel_body)
F_drag_world = Rm * drag_body

F_gravity = Matrix([0, 0, -m * g])

# Translation:  m a - F_thrust - F_gravity - F_drag = 0
acc_world = Matrix([diff(x, t, 2), diff(y, t, 2), diff(z, t, 2)])
nEq = m * acc_world - F_thrust_world - F_gravity - F_drag_world

# Rotation (diagonal inertia approximation)
ang_acc = Matrix([diff(alpha, t, 2), diff(beta, t, 2), diff(psi, t, 2)])
inertia_diag = Matrix([Ix * ang_acc[0], Iy * ang_acc[1], Iz * ang_acc[2]])
torques = Matrix([T1, T2, T3])
eEq = inertia_diag - torques

print("Translation equations (3):")
display(nEq)
print("Rotation equations (3):")
display(eEq)

In [ ]:
# ==== First-order form ====
x_dot     = Function('x_dot')(t)
y_dot     = Function('y_dot')(t)
z_dot     = Function('z_dot')(t)
alpha_dot = Function(r'\alpha_dot')(t)
beta_dot  = Function(r'\beta_dot')(t)
psi_dot   = Function(r'\psi_dot')(t)

state = Matrix([
    x, y, z,
    alpha, beta, psi,
    x_dot, y_dot, z_dot,
    alpha_dot, beta_dot, psi_dot
])

first_order_subs = {
    diff(x, t): x_dot,        diff(y, t): y_dot,        diff(z, t): z_dot,
    diff(alpha, t): alpha_dot, diff(beta, t): beta_dot, diff(psi, t): psi_dot,
}

nEq_fo = nEq.subs(first_order_subs)
eEq_fo = eEq.subs(first_order_subs)

highest_derivs = [
    diff(x_dot, t), diff(y_dot, t), diff(z_dot, t),
    diff(alpha_dot, t), diff(beta_dot, t), diff(psi_dot, t)
]
sol_highest = solve(list(nEq_fo) + list(eEq_fo), highest_derivs, dict=True)[0]

state_rhs = Matrix([
    x_dot, y_dot, z_dot,
    alpha_dot, beta_dot, psi_dot,
    sol_highest[diff(x_dot, t)],
    sol_highest[diff(y_dot, t)],
    sol_highest[diff(z_dot, t)],
    sol_highest[diff(alpha_dot, t)],
    sol_highest[diff(beta_dot, t)],
    sol_highest[diff(psi_dot, t)],
])

print("State-space RHS  f(x,u)  in  dx/dt = f(x,u):")
display(state_rhs)

## 2. Linearization at hover and augmented system

We linearize around the **hover equilibrium**: zero angles, zero velocities, `F1 = m·g`, zero torques.
The Jacobians `A = ∂f/∂x` and `B = ∂f/∂u` give the LTI plant `δẋ = A·δx + B·δu`.

To achieve **zero steady-state position error** under modeling errors and persistent disturbances
(e.g. drag), we augment the state with three position-error integrators `(∫x, ∫y, ∫z)`,
yielding a 15-dimensional system on which LQR is designed.


In [ ]:
# ==== Jacobians ====
control_vec = Matrix([F1, T1, T2, T3])
A = state_rhs.jacobian(state)
B = state_rhs.jacobian(control_vec)
print("A:", A.shape, "  B:", B.shape)

In [ ]:
# ==== Hover equilibrium (derived, not hardcoded) ====
# An equilibrium is a state where dx/dt = 0. We set the physical state to
# zero (no position, angles, or velocities — pure hover) and solve the
# state RHS for the inputs that make the dynamics vanish.
state_at_zero = {
    alpha: 0, beta: 0, psi: 0,
    x_dot: 0, y_dot: 0, z_dot: 0,
    alpha_dot: 0, beta_dot: 0, psi_dot: 0,
}

rhs_at_zero = state_rhs.subs(state_at_zero)

# Solve f(x=0, u) = 0 for u = (F1, T1, T2, T3).
hover_inputs = solve(rhs_at_zero, [F1, T1, T2, T3], dict=True)
assert len(hover_inputs) == 1, "Expected a unique hover equilibrium for the inputs"
hover_inputs = hover_inputs[0]
print("Hover equilibrium inputs (derived symbolically):")
display(hover_inputs)

# Build the substitution dict used to evaluate A and B at the equilibrium.
equilibrium_subs = {**state_at_zero, **hover_inputs}

A_eq = A.subs(equilibrium_subs)
B_eq = B.subs(equilibrium_subs)
print("A at hover:")
display(A_eq)
print("B at hover:")
display(B_eq)

In [ ]:
# ==== Augmented system (15 states) ====
# Position integrators: d/dt[int_p] = pos_ref - pos. For LQR design (regulator
# around hover with no reference), this reduces to d/dt[int_p] = -pos.

C = zeros(3, 12)
C[0, 0] = 1; C[1, 1] = 1; C[2, 2] = 1   # extract position

A_e = zeros(15, 15)
A_e[:12, :12] = A_eq
A_e[12:15, :12] = -C

B_e = zeros(15, 4)
B_e[:12, :] = B_eq

print("Augmented A_e:", A_e.shape, "  B_e:", B_e.shape)

## 3. LQR design

The LQR weights penalize position and roll heavily (we care about where we land and that
the rocket does not spin around its longitudinal axis), pitch/yaw moderately (we expect the
FF to drive them along the trajectory), and angular rates
heavily (no spinning allowed).


In [ ]:
# ==== Numerical parameters ====

params_subs = {
    m: 10,
    Ix: Rational(10, 3),
    Iy: Rational(10, 3),
    Iz: 1,
    g:  Rational(981, 100),    # 9.81
    c:  1,                     # lateral drag (body x, y)
    cz: Rational(2, 100),      # axial drag (body z), 0.02
    F1_max: 700,               # max trhuster force [N]
}

sim_Tf = 20 # simulation end time in seconds

A_e_num = np.array(A_e.subs(params_subs).evalf(), dtype=np.float64)
B_e_num = np.array(B_e.subs(params_subs).evalf(), dtype=np.float64)

In [ ]:
# ==== LQR ====
# State order: [x, y, z, alpha, beta, psi, x_dot, y_dot, z_dot,
#               alpha_dot, beta_dot, psi_dot, int_x, int_y, int_z]
Q = np.eye(15)
Q[0, 0] = 1000;  Q[1, 1] = 1000;  Q[2, 2]  = 1000   # position
Q[3, 3] = 100;   Q[4, 4] = 100;   Q[5, 5]  = 1000   # angles (roll heavy: penalize axial spin)
Q[6, 6] = 1;     Q[7, 7] = 1;     Q[8, 8]  = 1      # linear vel
Q[9, 9] = 1000;  Q[10, 10] = 1000; Q[11, 11] = 1000 # ang vel
Q[12, 12] = 100; Q[13, 13] = 100;  Q[14, 14] = 100  # integrators

R = np.eye(4)

K_e, S, E = ct.lqr(A_e_num, B_e_num, Q, R)
print("K_e shape:", K_e.shape)
print("Closed-loop eigenvalues (should all have negative real part):")
print(E)
assert np.all(E.real < 0), "LQR closed-loop is unstable — check Q, R or model"

## 4. Sanity check — small perturbation about hover

We lambdify the symbolic dynamics and run a closed-loop simulation with a small initial
pitch perturbation `α(0) = 0.1 rad` (≈5.7°), no reference trajectory. The augmented LQR
is the only controller. This validates the linearization, the integrators, and the gain matrix.


In [ ]:
# ==== Lambdify dynamics ====
state_rhs_num = state_rhs.subs(params_subs)

state_vars = [x, y, z, alpha, beta, psi,
              x_dot, y_dot, z_dot,
              alpha_dot, beta_dot, psi_dot]
control_vars = [F1, T1, T2, T3]

dynamics_func = lambdify(
    [t] + state_vars + control_vars,
    state_rhs_num,
    modules='numpy'
)

# Test at hover: derivatives should be ~zero
test = dynamics_func(0.0, 0,0,0,0,0,0, 0,0,0,0,0,0, params_subs[m]*params_subs[g], 0,0,0)
print("Dynamics at hover (should be ~0):", np.array(test).flatten())

In [ ]:
# ==== Closed-loop LQR-only simulation ====
m_val = params_subs[m]
g_val = params_subs[g]
u_hover = np.array([m_val * g_val, 0.0, 0.0, 0.0])

F1_MIN = 0
F1_MAX = params_subs[F1_max]

def closed_loop_lqr_only(t_eval, state_aug):
    x_phys = state_aug[:12]
    u = u_hover - K_e @ state_aug
    u[0] = np.clip(u[0], F1_MIN, F1_MAX)
    dx_phys = np.array(dynamics_func(t_eval, *x_phys, *u)).flatten()
    d_int = -x_phys[:3]   # regulator: integrate -position
    return np.concatenate([dx_phys, d_int])

x0 = np.zeros(15)
x0[0] = 0   # x
x0[1] = 0   # y
x0[2] = 10   # z
x0[3] = 0.2   # alpha

sol_hover = solve_ivp(
    closed_loop_lqr_only, (0, sim_Tf), x0,
    t_eval=np.linspace(0, sim_Tf, 500),
    method='RK45', rtol=1e-8, atol=1e-10
)

print("Status:", sol_hover.message)
print(f"Final  pos = {sol_hover.y[:3, -1]}")
print(f"Final  ang = {sol_hover.y[3:6, -1]} rad")

In [ ]:
# ==== Plot ====
labels = [
    'x [m]', 'y [m]', 'z [m]',
    'α [rad]', 'β [rad]', 'ψ [rad]',
    'ẋ [m/s]', 'ẏ [m/s]', 'ż [m/s]',
    'α̇ [rad/s]', 'β̇ [rad/s]', 'ψ̇ [rad/s]',
    '∫x [m·s]', '∫y [m·s]', '∫z [m·s]'
]
fig, axes = plt.subplots(5, 3, figsize=(13, 11), sharex=True)
axes = axes.flatten()
for i, ax in enumerate(axes):
    ax.plot(sol_hover.t, sol_hover.y[i], linewidth=1.3)
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
for ax in axes[-3:]:
    ax.set_xlabel('t [s]')
fig.suptitle('Closed-loop LQR (regulator), α(0) = 0.2 rad, z(0) = 10 m', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Feedforward via differential flatness

The 6DOF rocket is differentially flat in the position outputs `(x, y, z)` (roll is independently
flat). Given a reference acceleration `(ẍ_ref, ÿ_ref, z̈_ref)`, the corresponding thrust magnitude
and tilt angles are obtained algebraically:

- `F1_ff = m · ‖[ẍ, ÿ, z̈+g]‖`
- `α_ff  = atan2(ẍ, z̈+g)`
- `β_ff  = atan2(-ÿ, √(ẍ² + (z̈+g)²))`
- `ψ_ff  = 0`  (roll is held at zero by the feedforward)

Using **`atan2`** instead of `arccos`/`arcsin` gives
**full 4-quadrant coverage**.
The torques `T1_ff, T2_ff` follow from `Ix·α̈_ff, Iy·β̈_ff`.

Note: this FF **ignores aerodynamic drag**. The LQR is left to compensate for it — the cost
of this simplification is discussed in §8.


In [ ]:
# ==== FF via flatness (atan2 form) ====
x_ddot_ref, y_ddot_ref, z_ddot_ref = symbols(
    r'\ddot{x}_{ref} \ddot{y}_{ref} \ddot{z}_{ref}', real=True
)

F1_ff    = m * sqrt(x_ddot_ref**2 + y_ddot_ref**2 + (z_ddot_ref + g)**2)
alpha_ff = atan2(x_ddot_ref, z_ddot_ref + g)
beta_ff  = atan2(-y_ddot_ref, sqrt(x_ddot_ref**2 + (z_ddot_ref + g)**2))
psi_ff   = 0

print("F1_ff:");    display(F1_ff)
print("alpha_ff:"); display(alpha_ff)
print("beta_ff:");  display(beta_ff)

## 6. Reference trajectory — degree-4 polynomials

Per axis we impose 5 boundary conditions:

| condition | x | y | z |
|---|---|---|---|
| `pos(0)`  | -50  | 50  | 150  |
| `vel(0)`  | 0    | 5   | -50  |
| `pos(TF)` | 0    | 0   | 0    |
| `vel(TF)` | 0    | 0   | 0    |
| `acc(TF)` | 0    | 0   | 0    |

with `TF = 10 s`. Five conditions ⇒ **degree-4 polynomial** (5 coefficients) per axis.
The terminal acceleration constraint enforces a smooth landing.

The position/velocity magnitudes are chosen so that `F1_ff` stays within the actuator
limits `[F1_min, F1_max] = [0.1·m·g, 5·m·g]` for the entire descent.


In [ ]:
# ==== Polynomial trajectory: 5 BCs per axis -> degree 4 ====
a0, a1, a2, a3, a4 = symbols('a0 a1 a2 a3 a4', real=True)
b0, b1, b2, b3, b4 = symbols('b0 b1 b2 b3 b4', real=True)
c0_, c1_, c2_, c3_, c4_ = symbols('c0 c1 c2 c3 c4', real=True)

Tj_x = a4*t**4 + a3*t**3 + a2*t**2 + a1*t + a0
Tj_y = b4*t**4 + b3*t**3 + b2*t**2 + b1*t + b0
Tj_z = c4_*t**4 + c3_*t**3 + c2_*t**2 + c1_*t + c0_

Tj_x_dot,  Tj_y_dot,  Tj_z_dot  = diff(Tj_x, t),    diff(Tj_y, t),    diff(Tj_z, t)
Tj_x_ddot, Tj_y_ddot, Tj_z_ddot = diff(Tj_x, t, 2), diff(Tj_y, t, 2), diff(Tj_z, t, 2)

# Boundary conditions
xIn, yIn, zIn = -50, 50, 150
vxIn, vyIn, vzIn = 0, 5, -50
TF_val = sim_Tf

constraints_x = [
    Tj_x.subs(t, 0) - xIn,
    Tj_x.subs(t, TF_val) - 0,
    Tj_x_dot.subs(t, 0) - vxIn,
    Tj_x_dot.subs(t, TF_val) - 0,
    Tj_x_ddot.subs(t, TF_val) - 0,
]
constraints_y = [
    Tj_y.subs(t, 0) - yIn,
    Tj_y.subs(t, TF_val) - 0,
    Tj_y_dot.subs(t, 0) - vyIn,
    Tj_y_dot.subs(t, TF_val) - 0,
    Tj_y_ddot.subs(t, TF_val) - 0,
]
constraints_z = [
    Tj_z.subs(t, 0) - zIn,
    Tj_z.subs(t, TF_val) - 0,
    Tj_z_dot.subs(t, 0) - vzIn,
    Tj_z_dot.subs(t, TF_val) - 0,
    Tj_z_ddot.subs(t, TF_val) - 0,
]

coeffs_x = solve(constraints_x, [a4, a3, a2, a1, a0])
coeffs_y = solve(constraints_y, [b4, b3, b2, b1, b0])
coeffs_z = solve(constraints_z, [c4_, c3_, c2_, c1_, c0_])

print("Trajectory coefficients (x, y, z):")
print({k: float(v) for k, v in coeffs_x.items()})
print({k: float(v) for k, v in coeffs_y.items()})
print({k: float(v) for k, v in coeffs_z.items()})

In [ ]:
# ==== Substitute coefficients into FF and lambdify ====
all_subs = {**coeffs_x, **coeffs_y, **coeffs_z, **params_subs}

# Numerical references (functions of t)
Tj_x_num      = Tj_x.subs(all_subs)
Tj_y_num      = Tj_y.subs(all_subs)
Tj_z_num      = Tj_z.subs(all_subs)
Tj_x_dot_num  = Tj_x_dot.subs(all_subs)
Tj_y_dot_num  = Tj_y_dot.subs(all_subs)
Tj_z_dot_num  = Tj_z_dot.subs(all_subs)
Tj_x_ddot_num = Tj_x_ddot.subs(all_subs)
Tj_y_ddot_num = Tj_y_ddot.subs(all_subs)
Tj_z_ddot_num = Tj_z_ddot.subs(all_subs)

# FF as functions of t
traj_accel_subs = {
    x_ddot_ref: Tj_x_ddot_num,
    y_ddot_ref: Tj_y_ddot_num,
    z_ddot_ref: Tj_z_ddot_num,
}
F1_ff_num    = F1_ff.subs(traj_accel_subs).subs(params_subs)
alpha_ff_num = alpha_ff.subs(traj_accel_subs).subs(params_subs)
beta_ff_num  = beta_ff.subs(traj_accel_subs).subs(params_subs)

# Reference angular velocities and torques
alpha_ff_dot_num = diff(alpha_ff_num, t)
beta_ff_dot_num  = diff(beta_ff_num, t)
T1_ff_num = (Ix * diff(alpha_ff_num, t, 2)).subs(params_subs)
T2_ff_num = (Iy * diff(beta_ff_num, t, 2)).subs(params_subs)

# Lambdify
Tj_x_func        = lambdify(t, Tj_x_num,       modules='numpy')
Tj_y_func        = lambdify(t, Tj_y_num,       modules='numpy')
Tj_z_func        = lambdify(t, Tj_z_num,       modules='numpy')
Tj_x_dot_func    = lambdify(t, Tj_x_dot_num,   modules='numpy')
Tj_y_dot_func    = lambdify(t, Tj_y_dot_num,   modules='numpy')
Tj_z_dot_func    = lambdify(t, Tj_z_dot_num,   modules='numpy')
F1_ff_func       = lambdify(t, F1_ff_num,      modules='numpy')
alpha_ff_func    = lambdify(t, alpha_ff_num,   modules='numpy')
beta_ff_func     = lambdify(t, beta_ff_num,    modules='numpy')
alpha_ff_dot_func= lambdify(t, alpha_ff_dot_num, modules='numpy')
beta_ff_dot_func = lambdify(t, beta_ff_dot_num,  modules='numpy')
T1_ff_func       = lambdify(t, T1_ff_num,      modules='numpy')
T2_ff_func       = lambdify(t, T2_ff_num,      modules='numpy')

def x_ref_at(t_val):
    """15-dim reference state at time t."""
    return np.array([
        Tj_x_func(t_val), Tj_y_func(t_val), Tj_z_func(t_val),
        alpha_ff_func(t_val), beta_ff_func(t_val), 0.0,
        Tj_x_dot_func(t_val), Tj_y_dot_func(t_val), Tj_z_dot_func(t_val),
        alpha_ff_dot_func(t_val), beta_ff_dot_func(t_val), 0.0,
        0.0, 0.0, 0.0
    ])

print(f"F1_ff(0)    = {F1_ff_func(0):.2f} N      (limits: [{F1_MIN:.1f}, {F1_MAX:.1f}])")
print(f"alpha_ff(0) = {np.degrees(alpha_ff_func(0)):.3f} deg")
print(f"beta_ff(0)  = {np.degrees(beta_ff_func(0)):.3f} deg")

In [ ]:
# ==== Plot FF along the trajectory ====
t_grid = np.linspace(0, TF_val, 500)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(t_grid, F1_ff_func(t_grid), linewidth=1.5)
axes[0].axhline(F1_MIN, color='r', linestyle='--', alpha=0.6, label='F1_min')
axes[0].axhline(F1_MAX, color='r', linestyle='--', alpha=0.6, label='F1_max')
axes[0].set_ylabel('F₁_ff [N]'); axes[0].set_xlabel('t [s]')
axes[0].set_title('Feedforward thrust'); axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(t_grid, np.degrees(alpha_ff_func(t_grid)), linewidth=1.5)
axes[1].set_ylabel('α_ff [deg]'); axes[1].set_xlabel('t [s]')
axes[1].set_title('Pitch FF'); axes[1].grid(True, alpha=0.3)

axes[2].plot(t_grid, np.degrees(beta_ff_func(t_grid)), linewidth=1.5)
axes[2].set_ylabel('β_ff [deg]'); axes[2].set_xlabel('t [s]')
axes[2].set_title('Yaw FF'); axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Closed-loop tracking — best case (no drag)

We close the loop with **FF + LQR + anti-windup** and integrate the dynamics with **drag
turned off** (`c = c_z = 0`). This is the *best case* and validates that:

- the FF expressions are consistent with the dynamics,
- the LQR correction on tracking error is well-behaved,
- the anti-windup logic (freeze position integrators when `F1` saturates) prevents the
  divergence we observed without it.

Initial condition is set exactly on the reference trajectory; tracking error should
decay to numerical zero.


In [ ]:
# ==== No-drag dynamics (only for the best-case test in this section) ====
params_subs_no_drag = {**params_subs, c: 0, cz: 0}
state_rhs_no_drag = state_rhs.subs(params_subs_no_drag)
dynamics_func_no_drag = lambdify(
    [t] + state_vars + control_vars,
    state_rhs_no_drag,
    modules='numpy'
)

def closed_loop_no_drag(t_eval, state_aug):
    x_phys = state_aug[:12]
    x_ref = x_ref_at(t_eval)
    err = state_aug - x_ref

    u_ff = np.array([
        F1_ff_func(t_eval),
        T1_ff_func(t_eval),
        T2_ff_func(t_eval),
        0.0,
    ])
    u_lqr = -K_e @ err
    u = u_ff + u_lqr

    # Saturate F1 and detect saturation for anti-windup
    u0_unsat = u[0]
    u0_sat = np.clip(u0_unsat, F1_MIN, F1_MAX)
    f1_saturated = (u0_unsat != u0_sat)
    u[0] = u0_sat

    dx_phys = np.array(dynamics_func_no_drag(t_eval, *x_phys, *u)).flatten()

    # Anti-windup: freeze integrators when F1 saturates
    if f1_saturated:
        d_int = np.zeros(3)
    else:
        d_int = -(x_phys[:3] - x_ref[:3])

    return np.concatenate([dx_phys, d_int])


x0_best = x_ref_at(0.0)
sol_best = solve_ivp(
    closed_loop_no_drag, (0, TF_val), x0_best,
    t_eval=np.linspace(0, TF_val, 1000),
    method='RK45', rtol=1e-8, atol=1e-10
)
print("Status:", sol_best.message)
print(f"Final  pos = [{sol_best.y[0,-1]:9.2e}, {sol_best.y[1,-1]:9.2e}, {sol_best.y[2,-1]:9.2e}] m")
print(f"Final  ang = [{np.degrees(sol_best.y[3,-1]):8.4f}, "
      f"{np.degrees(sol_best.y[4,-1]):8.4f}, "
      f"{np.degrees(sol_best.y[5,-1]):8.4f}] deg")

In [ ]:
# ==== Visualize tracking ====
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

t_ref_grid = np.linspace(0, TF_val, 500)
x_ref_traj = Tj_x_func(t_ref_grid)
y_ref_traj = Tj_y_func(t_ref_grid)
z_ref_traj = Tj_z_func(t_ref_grid)

# 3D
fig = plt.figure(figsize=(9, 6.5))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.plot(x_ref_traj, y_ref_traj, z_ref_traj, 'r--', lw=2, alpha=0.7, label='Reference')
ax3d.plot(sol_best.y[0], sol_best.y[1], sol_best.y[2], 'b-', lw=1.5, label='Simulated')
ax3d.scatter(*x0_best[:3], color='green', s=70, marker='o', label='Start')
ax3d.scatter(0, 0, 0, color='red', s=70, marker='x', label='Target')
ax3d.set_xlabel('x [m]'); ax3d.set_ylabel('y [m]'); ax3d.set_zlabel('z [m]')
ax3d.set_title('3D trajectory — FF + LQR + anti-windup, no drag')
ax3d.legend()
plt.show()

# Position tracking
fig, axes = plt.subplots(3, 1, figsize=(9.5, 6.5), sharex=True)
for i, name in enumerate(['x', 'y', 'z']):
    ref_vals = [x_ref_traj, y_ref_traj, z_ref_traj][i]
    axes[i].plot(t_ref_grid, ref_vals, 'r--', lw=2, alpha=0.7, label='Reference')
    axes[i].plot(sol_best.t, sol_best.y[i], 'b-', lw=1.4, label='Simulated')
    axes[i].set_ylabel(f'{name} [m]')
    axes[i].grid(True, alpha=0.3); axes[i].legend(loc='best')
axes[-1].set_xlabel('t [s]')
fig.suptitle('Position tracking — best case (no drag)')
fig.tight_layout()
plt.show()

In [ ]:
# ==== Verify the LQR is doing something (its contribution should be tiny here) ====
u_lqr_history = np.zeros((4, len(sol_best.t)))
for i, t_i in enumerate(sol_best.t):
    err_i = sol_best.y[:, i] - x_ref_at(t_i)
    u_lqr_history[:, i] = -K_e @ err_i

fig, axes = plt.subplots(2, 2, figsize=(9.5, 4.5))
labels = ['u_lqr F₁ [N]', 'u_lqr T₁ [N·m]', 'u_lqr T₂ [N·m]', 'u_lqr T₃ [N·m]']
for i, ax in enumerate(axes.flatten()):
    ax.plot(sol_best.t, u_lqr_history[i])
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)
fig.suptitle('LQR contribution over time — no drag')
fig.tight_layout()
plt.show()

## 8. Realistic test — drag active and initial perturbation

We now run the same closed loop but with **drag enabled in the dynamics** (the FF still ignores
it — see §9) and a **small initial perturbation** off the reference:

- `Δpos = (5, -5, 10) m`
- `Δvel = (0.5, 0.5, -1) m/s`
- `Δα = Δβ = 2° ≈ 0.035 rad`

This is the configuration the C++/WASM runtime is expected to face.
The LQR is left to compensate the unmodelled drag plus the initial mismatch.


In [ ]:
# ==== Closed loop with drag and IC perturbation ====
def closed_loop_realistic(t_eval, state_aug):
    x_phys = state_aug[:12]
    x_ref = x_ref_at(t_eval)
    err = state_aug - x_ref

    u_ff = np.array([
        F1_ff_func(t_eval),
        T1_ff_func(t_eval),
        T2_ff_func(t_eval),
        0.0,
    ])
    u_lqr = -K_e @ err
    u = u_ff + u_lqr

    u0_unsat = u[0]
    u0_sat = np.clip(u0_unsat, F1_MIN, F1_MAX)
    f1_saturated = (u0_unsat != u0_sat)
    u[0] = u0_sat

    # Full drag dynamics here
    dx_phys = np.array(dynamics_func(t_eval, *x_phys, *u)).flatten()

    if f1_saturated:
        d_int = np.zeros(3)
    else:
        d_int = -(x_phys[:3] - x_ref[:3])

    return np.concatenate([dx_phys, d_int])


# Initial condition: reference + small perturbation
x0_real = x_ref_at(0.0).copy()
x0_real[0] += 5.0          # Δx
x0_real[1] += -5.0         # Δy
x0_real[2] += 10.0           # Δz
x0_real[6] += 0.5          # Δẋ
x0_real[7] += 0.5          # Δẏ
x0_real[8] += -1.0         # Δż
x0_real[3] += np.deg2rad(2)   # Δα
x0_real[4] += np.deg2rad(2)   # Δβ

sol_real = solve_ivp(
    closed_loop_realistic, (0, TF_val), x0_real,
    t_eval=np.linspace(0, TF_val, 1000),
    method='RK45', rtol=1e-7, atol=1e-9
)

print("Status:", sol_real.message)
print(f"\nFinal state at t = {sol_real.t[-1]:.2f} s:")
print(f"  pos  = [{sol_real.y[0,-1]:8.3f}, {sol_real.y[1,-1]:8.3f}, {sol_real.y[2,-1]:8.3f}] m")
print(f"  ang  = [{np.degrees(sol_real.y[3,-1]):7.2f}, "
      f"{np.degrees(sol_real.y[4,-1]):7.2f}, "
      f"{np.degrees(sol_real.y[5,-1]):7.2f}] deg")
print(f"  vel  = [{sol_real.y[6,-1]:8.3f}, {sol_real.y[7,-1]:8.3f}, {sol_real.y[8,-1]:8.3f}] m/s")

# Position-tracking error norm at the end
final_pos_err = np.linalg.norm(sol_real.y[:3, -1] - np.array([0., 0., 0.]))
print(f"\nFinal position error from target (0,0,0): {final_pos_err:.3f} m")

In [ ]:
# ==== Plots: tracking error and 3D ====
err_pos = sol_real.y[:3] - np.array([x_ref_at(ti)[:3] for ti in sol_real.t]).T
err_ang = sol_real.y[3:6] - np.array([x_ref_at(ti)[3:6] for ti in sol_real.t]).T

fig, axes = plt.subplots(2, 3, figsize=(13, 5.5))
for i, name in enumerate(['x', 'y', 'z']):
    axes[0, i].plot(sol_real.t, err_pos[i], lw=1.4)
    axes[0, i].set_ylabel(f'error {name} [m]')
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].axhline(0, color='k', lw=0.5, alpha=0.5)
for i, name in enumerate(['α', 'β', 'ψ']):
    axes[1, i].plot(sol_real.t, np.degrees(err_ang[i]), lw=1.4)
    axes[1, i].set_ylabel(f'error {name} [deg]')
    axes[1, i].set_xlabel('t [s]')
    axes[1, i].grid(True, alpha=0.3)
    axes[1, i].axhline(0, color='k', lw=0.5, alpha=0.5)
fig.suptitle('Tracking error — drag ACTIVE + initial perturbation')
fig.tight_layout()
plt.show()

# 3D plot
fig = plt.figure(figsize=(9, 6.5))
ax3d = fig.add_subplot(111, projection='3d')
ax3d.plot(x_ref_traj, y_ref_traj, z_ref_traj, 'r--', lw=2, alpha=0.7, label='Reference')
ax3d.plot(sol_real.y[0], sol_real.y[1], sol_real.y[2], 'b-', lw=1.5, label='Simulated')
ax3d.scatter(*x0_real[:3], color='green', s=70, marker='o', label='Start (perturbed)')
ax3d.scatter(0, 0, 0, color='red', s=70, marker='x', label='Target')
ax3d.set_xlabel('x [m]'); ax3d.set_ylabel('y [m]'); ax3d.set_zlabel('z [m]')
ax3d.set_title('3D trajectory — drag ACTIVE + IC perturbation')
ax3d.legend()
plt.show()

In [ ]:
# ==== LQR contribution over time (drag active) ====
# In the realistic case, the LQR has actual work to do: it has to compensate
# the unmodelled drag force and pull the state back from the initial
# perturbation. Here we expect non-trivial contributions on all four channels,
# unlike the best-case plot in §7 where the LQR was essentially idle.
u_lqr_real = np.zeros((4, len(sol_real.t)))
for i, t_i in enumerate(sol_real.t):
    err_i = sol_real.y[:, i] - x_ref_at(t_i)
    u_lqr_real[:, i] = -K_e @ err_i

fig, axes = plt.subplots(2, 2, figsize=(9.5, 4.5))
labels = ['u_lqr F₁ [N]', 'u_lqr T₁ [N·m]', 'u_lqr T₂ [N·m]', 'u_lqr T₃ [N·m]']
for i, ax in enumerate(axes.flatten()):
    ax.plot(sol_real.t, u_lqr_real[i], lw=1.4)
    ax.set_ylabel(labels[i])
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
fig.suptitle('LQR contribution over time — drag ACTIVE + IC perturbation')
fig.tight_layout()
plt.show()

## 9. Discussion and limitations

This section documents the **known limits** of the design above. They are deliberate
trade-offs, not bugs.

### 9.1 LQR is linearized about hover, not along the trajectory

`A_e, B_e, K_e` are computed at the hover equilibrium `(angles=0, velocities=0, F1=m·g)`.
The descent trajectory leaves this neighbourhood.
As the state moves further from hover, the LQR gain becomes **suboptimal** (still stabilizing in
the basin of attraction we use, but not the LQR-optimal gain for the actual operating point).

The textbook fix is **TVLQR** (time-varying LQR computed by integrating the differential
Riccati equation backwards along the reference). It would tighten tracking, especially
near the high-velocity portion of the descent, but is out of scope for the current phase.
Gain scheduling against `vz` would be an intermediate option.

### 9.2 Feedforward ignores aerodynamic drag — and this is the dominant error source

Drag in our model is body-frame: `F_drag_body = -diag(c, c, c_z) · v_body`,
i.e. `F_drag_world = Rm · diag(c,c,c_z) · Rmᵀ · v_world`.

A **drag-aware FF** would need to invert this expression along the trajectory: given desired
position `p_ref(t)` and velocity `v_ref(t)`, find `(F1_ff, α_ff, β_ff)` such that
`m · p̈_ref = F_thrust_world(F1, α, β) + F_gravity + F_drag_world(α, β, v_ref)`.

Because the drag depends on `(α, β)` through `Rm`, this is an **implicit** equation in the FF
controls — it has no closed form and requires either (A) a fixed-point iteration at every
time step, or (B) a simplifying assumption (e.g. world-frame diagonal drag), neither of
which is satisfying. We therefore keep the simple FF and **delegate drag rejection to the LQR
plus the position integrators**, accepting some steady-state offset.

The cost of this simplification is visible in §8: the same closed loop that lands at
machine precision without drag (§7) lands ~20 cm off when drag is active. Most of this
error comes from the LQR steady-state response to the unmodelled drag force, not from
the initial perturbation (we verified this by toggling each independently).

### 9.3 Drag is modelled as linear in velocity, not quadratic

Real aerodynamic drag scales as `½·ρ·C_d·A·|v|·v` — quadratic in the speed and acting along
the direction of motion. Our model uses the **linear** approximation `F_drag = -c·v` (in
body frame), valid only at low Reynolds numbers (viscous regime) and clearly not the right
physics for a descending rocket.

Two consequences worth noting:

- **Linearization at hover is asymmetric for the two models.** With linear drag, the
  Jacobian `A` at hover (where `v = 0`) inherits non-zero diagonal terms `-c/m` from the
  drag, which shape the LQR gain. With quadratic drag, the derivative `∂(|v|·v)/∂v` at
  `v = 0` is zero, so the drag would **disappear** from the linearization — the LQR would
  see a drag-free plant near hover and have to rely entirely on the augmented integrators
  to reject the actual quadratic drag away from hover. Either way, the LQR design is not
  significantly informed by the drag model.

- **Fidelity of the simulated truth.** For descent speeds up to ~50 m/s and a 10 kg
  vehicle, neither model is realistic — but the linear one is a **simpler stand-in** that
  produces qualitatively the right behaviour (forces opposing motion, growing with speed)
  without complicating the symbolic FF derivation. Switching to quadratic drag would
  break the closed-form flatness expressions in §5 (`F1_ff` and the tilt angles would no
  longer be algebraic functions of `(ẍ, ÿ, z̈+g)`).

For a portfolio iteration focused on the **structure of the controller** (FF + LQR +
anti-windup), the linear-drag approximation is acceptable; in a follow-up that targets
realistic flight envelopes, the natural step is quadratic drag plus drag-aware FF (see §9.2).

### 9.4 No quaternions, no attitude wrapping

Pitch (`α`) and yaw (`β`) are tracked as `Function(t)` in `(-π, π]` via `atan2`, and roll (`ψ`)
is held at zero by the FF. For aggressive trajectories or extended simulations, `α` or `β` can wrap past `±π`,
which would confuse the linearized error feedback. A quaternion-based attitude representation
is the right long-term fix and is left for a future iteration.

### 9.5 Other deliberate omissions

- **Trajectory optimization** (acados, CasADi): the polynomial reference is not
  optimal in any sense; it satisfies smooth boundary conditions, that is all.
- **DAE solver / contact model**: the simulator does not stop at `z = 0`
- **Sensor noise / state estimator**: the closed loop uses full state feedback. Adding a
  Kalman filter on noisy IMU/GPS would be a natural next step.

### 9.6 Bottom line

For this portfolio iteration, **FF (no drag) + LQR + anti-windup** lands within ~0.2 m of
the target with bounded attitude excursions, despite drag and a non-trivial initial
perturbation. Improving on this would require drag-aware FF (closing the dominant error
source) or TVLQR (tightening the linearization assumption) — both deliberately deferred.


## 10. C++ code export for the runtime

This section generates a ready-to-include C++ class that the C++/WASM simulator
consumes. The class encapsulates the model and the controller (`FF + LQR`) for
this specific configuration; the runtime owns the trajectory, the integrator, and
the simulation loop.

The generated artifact is a single `.hpp / .cpp` pair, e.g. `rocket_ff_lqr_01.hpp/.cpp`,
with a class `CDS::Rocket::FF_LQR_01`. Its public surface:

- `ExecuteControl(state, reference) -> u`  — total control input `u_ff + u_lqr`,
  with the thrust channel saturated. The `Reference` is a struct with `pos, vel,
  acc, jerk, snap` (3-vec each) — the `acc` drives the kinematic FF, `jerk` and
  `snap` drive the torque FF.
- `Dynamics(state, u, ref_pos, userF) -> dxdt`  — the 6DOF nonlinear RHS,
  with augmented integrators driven by `ref_pos`. The runtime feeds this into
  its own RK4 step.
- `Get/SetState(s, StateName)`, `Get/SetParam(ParamName)`  — type-safe accessors.
  `StateToIdx(StateName)` is a public `constexpr` helper used internally and
  available to clients.

The trajectory polynomial coefficients are **not** baked into the generated class
— the model is decoupled from any specific trajectory. The runtime owns a
`Trajectory` object (Poly4, spline, …) that produces a `Reference` at each step.

The generation logic lives in the reusable module
[`descent_codegen.py`](./descent_codegen.py). It applies a small symbolic
pipeline before printing so the output reads like hand-written code:

- the dynamics RHS is `trigsimp`-ed and then `expand_trig`-ed so that roll (`ψ`)
  drops out of the lateral drag terms (it cancels exactly via `Rm·diag·Rmᵀ` —
  the lateral drag tensor is isotropic in body x-y),
- attitude trig functions are aliased to short locals (`sa, ca, sb, cb, …`) and
  precomputed once per call — only the ones that actually appear in the equations,
- terms are collected by `F1, xd, yd, zd` to expose the structure of the model,
- `pow(x, 2)` is rewritten as `x*x` for readability,
- the LQR gain matrix is emitted as a `static constexpr std::array` literal.


In [ ]:
import sys, os
# Make descent_codegen importable from the notebook directory.
if 'descent_codegen' in sys.modules:
    del sys.modules['descent_codegen']  # force reload during interactive work
sys.path.insert(0, '.')
from descent_codegen import DescentCodegen, CodegenConfig

# We need a *symbolic* state RHS with m, c, g, F1_max, ... still as SymPy
# symbols — the codegen replaces them with `m_p.<field>` accesses in the
# generated C++. (params_subs in earlier cells substituted them with floats
# for numeric work; here we want them symbolic.)

# Already declared in Section 1: m, Ix, Iy, Iz, g, F1_max, c, cz, and
# alpha, beta, psi (state), F1, T1, T2, T3 (controls), x, y, z, x_dot, ...

# Symbols for user-input forces (inertial frame, added to the dynamics).
userFx, userFy, userFz = sp.symbols('userFx userFy userFz', real=True)

# Symbols for the reference acceleration / jerk / snap that drive the FF.
ax_sym, ay_sym, az_sym = sp.symbols('ax ay az', real=True)
jx_sym, jy_sym, jz_sym = sp.symbols('jx jy jz', real=True)
sx_sym, sy_sym, sz_sym = sp.symbols('sx sy sz', real=True)

# ----- Static-form FF expressions in (ax, ay, az, jx, jy, jz, sx, sy, sz) -----
# Kinematic FF (closed form in acc + g).
F1_ff_static    = m * sp.sqrt(ax_sym**2 + ay_sym**2 + (az_sym + g)**2)
alpha_ff_static = sp.atan2(ax_sym, az_sym + g)
beta_ff_static  = sp.atan2(-ay_sym, sp.sqrt(ax_sym**2 + (az_sym + g)**2))

# Torque FF: T1_ff = Ix * d^2(alpha_ff)/dt^2, T2_ff = Iy * d^2(beta_ff)/dt^2.
# Build them by chaining time derivatives on a time-dependent stand-in, then
# rewrite the diff(...) with the corresponding jerk / snap symbols so the
# final expression is purely algebraic in (a, j, s).
ax_t = sp.Function('ax')(t); ay_t = sp.Function('ay')(t); az_t = sp.Function('az')(t)
alpha_ff_t = sp.atan2(ax_t, az_t + g)
beta_ff_t  = sp.atan2(-ay_t, sp.sqrt(ax_t**2 + (az_t + g)**2))
T1_ff_t = Ix * sp.diff(alpha_ff_t, t, 2)
T2_ff_t = Iy * sp.diff(beta_ff_t,  t, 2)

def _to_static(expr):
    """Replace diff(ax, t)->jx, diff(ax, t, 2)->sx, ax(t)->ax, etc."""
    e = expr
    e = e.subs({sp.diff(ax_t, t, 2): sx_sym,
                sp.diff(ay_t, t, 2): sy_sym,
                sp.diff(az_t, t, 2): sz_sym})
    e = e.subs({sp.diff(ax_t, t): jx_sym,
                sp.diff(ay_t, t): jy_sym,
                sp.diff(az_t, t): jz_sym})
    e = e.subs({ax_t: ax_sym, ay_t: ay_sym, az_t: az_sym})
    return sp.simplify(e)

T1_ff_static = _to_static(T1_ff_t)
T2_ff_static = _to_static(T2_ff_t)
print("Built static-form FF expressions (algebraic in ax, jx, sx, ...).")
print(f"  T1_ff has {sp.count_ops(T1_ff_static)} ops")
print(f"  T2_ff has {sp.count_ops(T2_ff_static)} ops")

In [ ]:
# ----- Drive the codegen -----
cfg = CodegenConfig(
    parent_namespace='CDS::Dynamics',
    model_name='FF_LQR_01',
    out_dir='exported_cpp',
    notebook_name='01_model_derivation.ipynb',
    author='Diego Perazzolo',
)
print(f"module name (derived): {cfg.module_name}")
print(f"will write: {cfg.out_dir}/{cfg.module_name}.hpp")
print(f"            {cfg.out_dir}/{cfg.module_name}.cpp")

cg = DescentCodegen(cfg)

# Symbol bindings — the codegen replaces these with array indices, struct
# fields, and the literals declared in the shared header.
cg.set_state_symbols([x, y, z, alpha, beta, psi,
                      x_dot, y_dot, z_dot,
                      alpha_dot, beta_dot, psi_dot])
cg.set_input_symbols([F1, T1, T2, T3])
# Order MUST match CodegenConfig.param_field_names = (m, Ix, Iy, Iz, g, c, cz, F1_max).
cg.set_physics_symbols([m, Ix, Iy, Iz, g, c, cz, F1_max])
cg.set_user_force_symbols([userFx, userFy, userFz])

# Mathematical content.
cg.set_dynamics(state_rhs)
cg.set_feedforward_kinematic(
    F1_ff_static, alpha_ff_static, beta_ff_static,
    ref_acc_syms=(ax_sym, ay_sym, az_sym),
)
cg.set_feedforward_torque(
    T1_ff_static, T2_ff_static,
    ref_acc_syms=(ax_sym, ay_sym, az_sym),
    ref_jerk_syms=(jx_sym, jy_sym, jz_sym),
    ref_snap_syms=(sx_sym, sy_sym, sz_sym),
)
cg.set_lqr_gain(K_e)

# Emit the .hpp/.cpp pair.
h_path, c_path = cg.write()
print(f"\nWrote {h_path} ({os.path.getsize(h_path)} bytes)")
print(f"Wrote {c_path} ({os.path.getsize(c_path)} bytes)")

In [ ]:
# ==== Generate the C++ body for Poly4::GetReference(t, ref) ====
# Emits the polynomial reference (pos, vel, acc, jerk, snap) along x, y, z,
# in Horner form, ready to be pasted inside the body of Poly4::GetReference.
import re

# Numerical polynomials in t: pos, vel, acc, jerk, snap for each axis.
poly_by_axis = {
    'x': Tj_x.subs(coeffs_x),
    'y': Tj_y.subs(coeffs_y),
    'z': Tj_z.subs(coeffs_z),
}

def horner(expr, var, prec=18):
    """Render `expr` (polynomial in var) in Horner form as a C string."""
    expr = sp.expand(expr)
    poly = expr.as_poly(var)
    if poly is None:
        return sp.ccode(sp.N(expr, prec))
    coeffs = [sp.N(c, prec) for c in poly.all_coeffs()]
    if len(coeffs) == 1:
        return sp.ccode(coeffs[0])
    out = sp.ccode(coeffs[0])
    var_name = str(var)
    for c in coeffs[1:]:
        c_str = sp.ccode(c)
        if c_str.startswith("-"):
            out = f"({out})*{var_name} - {c_str[1:]}"
        else:
            out = f"({out})*{var_name} + {c_str}"
    return out

# Build the function body
lines = []
lines.append("    const double t = time;  // local alias")
lines.append("")
labels = [("pos",  0), ("vel",  1), ("acc",  2), ("jerk", 3), ("snap", 4)]
for label, order in labels:
    lines.append(f"    // {label}")
    for i, axis in enumerate(['x', 'y', 'z']):
        deriv = sp.diff(poly_by_axis[axis], t, order)
        body = horner(deriv, t)
        lines.append(f"    ref.{label}[{i}] = {body};")
    lines.append("")
lines.append("    return true;")

print("// ---- paste inside Poly4::GetReference(time, ref) ----")
print("\n".join(lines))


In [ ]:
# ==== Initial state at t=0 (for the C++ Rocket() constructor) ====
import numpy as np
x0 = x_ref_at(0.0)

state_names = [
    'X', 'Y', 'Z',
    'Alpha', 'Beta', 'Psi',
    'XDot', 'YDot', 'ZDot',
    'AlphaDot', 'BetaDot', 'PsiDot',
    'IntX', 'IntY', 'IntZ',
]

print("// Initial state on the reference trajectory at t=0")
print("// Paste inside Rocket::Rocket() — replace the existing IC block.\n")
print("using SN = CDS::Dynamics::FF_LQR_01::StateName;")
for name, value in zip(state_names, x0):
    # skip integrators (always 0 at start)
    if name.startswith('Int'):
        continue
    if abs(value) < 1e-15:
        v_str = "0.0"
    elif abs(value) < 1e-3:
        v_str = f"{value:.6e}"
    else:
        v_str = f"{value:.6f}"
    print(f"FF_LQR_01::SetState(m_state, SN::{name:9s}, {v_str});")

print()
print("// Numerical summary:")
print(f"  Position [m]:        ({x0[0]:8.3f}, {x0[1]:8.3f}, {x0[2]:8.3f})")
print(f"  Attitude [rad]:      ({x0[3]:8.3f}, {x0[4]:8.3f}, {x0[5]:8.3f})")
print(f"  Linear vel [m/s]:    ({x0[6]:8.3f}, {x0[7]:8.3f}, {x0[8]:8.3f})")
print(f"  Angular vel [rad/s]: ({x0[9]:8.3f}, {x0[10]:8.3f}, {x0[11]:8.3f})")


In [ ]:
# Show the generated header, so the API surface is visible inline.
with open(h_path) as f:
    print(f.read())

In [ ]:
# Show the generated Dynamics() body — to confirm it reads like hand-written code.
import re
with open(c_path) as f:
    src = f.read()
m = re.search(r'StateVec FF_LQR_01::Dynamics[\s\S]*?\n\}\n', src)
if m:
    print(m.group(0))

In [ ]:
# ==== Validate the export ====
# This cell compiles the generated C++ together with a small driver, runs it,
# and compares every value against the Python ground truth — a regression test
# for the codegen. If the host toolchain cannot compile vanilla C++ (which can
# happen on macOS when Xcode CLT and Homebrew GCC disagree on the standard
# library), the cell prints a warning and skips the run; the .hpp/.cpp files
# have already been written by the previous cell and remain the deliverable.
import subprocess, shutil

# Stand-in for the project's shared header. In the real C++/WASM project
# Vec3 / RefVec / UserForces / Reference are declared in core_defs.hpp;
# here we provide a minimal stand-in just so the generated file compiles.
core_defs = """#pragma once
#include <array>
namespace CDS {
    using Vec3 = std::array<double, 3>;
    using RefVec = Vec3;
    using UserForces = Vec3;
    struct Reference {
        Vec3 pos;
        Vec3 vel;
        Vec3 acc;
        Vec3 jerk;
        Vec3 snap;
    };
}
"""
with open('exported_cpp/core_defs.hpp', 'w') as f:
    f.write(core_defs)

driver = r"""
#include <cstdio>
#include "dynamics_ff_lqr_01.hpp"

int main() {
    using namespace CDS;
    using namespace CDS::Dynamics;

    FF_LQR_01 model;
    model.SetParam(FF_LQR_01::ParamName::Mass,        10.0);
    model.SetParam(FF_LQR_01::ParamName::Ix,          10.0/3.0);
    model.SetParam(FF_LQR_01::ParamName::Iy,          10.0/3.0);
    model.SetParam(FF_LQR_01::ParamName::Iz,          1.0);
    model.SetParam(FF_LQR_01::ParamName::Gravity,     9.81);
    model.SetParam(FF_LQR_01::ParamName::DragLateral, 1.0);
    model.SetParam(FF_LQR_01::ParamName::DragAxial,   0.02);
    model.SetParam(FF_LQR_01::ParamName::ThrustMax,   700.0);

    FF_LQR_01::StateVec s = {1.0, -2.0, 50.0,  0.05, -0.03, 0.0,
                             0.5, -0.1, -10.0,  0.01, -0.02, 0.0,
                             0.1, 0.2, -0.05};
    FF_LQR_01::InputVec u = {100.0, 0.5, -0.3, 0.0};
    RefVec ref_pos = {0.0, 0.0, 0.0};
    UserForces userF = {2.5, -1.5, 0.7};

    auto dxdt = model.Dynamics(s, u, ref_pos, userF);
    for (double v : dxdt) std::printf("%.15e\n", v);

    Reference r;
    r.pos  = {10.0, -5.0, 100.0};
    r.vel  = {2.0, -1.0, -8.0};
    r.acc  = {0.5, -0.2, -3.0};
    r.jerk = {0.1, -0.05, 0.2};
    r.snap = {0.01, -0.02, 0.03};
    auto u_ctrl = model.ExecuteControl(s, r);
    for (double v : u_ctrl) std::printf("%.15e\n", v);
    return 0;
}
"""
with open('exported_cpp/_driver.cpp', 'w') as f:
    f.write(driver)

# ---- Try to compile with whatever C++ compiler is available ----
def _try_compile():
    # Try g++ first, then clang++. macOS users may need clang++ + libc++.
    candidates = [
        ['g++',     '-O2', '-std=c++17', '-Wall', '-Wextra'],
        ['clang++', '-O2', '-std=c++17', '-stdlib=libc++', '-Wall', '-Wextra'],
        ['clang++', '-O2', '-std=c++17', '-Wall', '-Wextra'],
    ]
    for argv in candidates:
        if shutil.which(argv[0]) is None:
            continue
        cc = subprocess.run(
            argv + ['-Iexported_cpp',
                    'exported_cpp/dynamics_ff_lqr_01.cpp', 'exported_cpp/_driver.cpp',
                    '-o', 'exported_cpp/_driver'],
            capture_output=True, text=True,
        )
        if cc.returncode == 0:
            return argv[0], cc
        print(f"  [{argv[0]}] failed:")
        for line in cc.stderr.strip().splitlines()[:6]:
            print(f"      {line}")
    return None, None

print("Compiling generated C++...")
compiler, cc = _try_compile()
if compiler is None:
    print()
    print("WARNING: no working C++ compiler found on this host.")
    print("The .hpp / .cpp files were generated successfully by the previous cell")
    print("and are the deliverable. Skipping the C++ vs Python comparison.")
    print()
    print("To enable the regression check, install a working toolchain:")
    print("  - Linux:  apt install g++   (or equivalent)")
    print("  - macOS:  xcode-select --install   (uses clang++ + libc++)")
else:
    print(f"Compiled with: {compiler}")
    if cc.stderr:
        print("Compiler warnings:")
        print(cc.stderr[:1000])

    run = subprocess.run(['exported_cpp/_driver'], capture_output=True, text=True)
    c_vals = [float(line) for line in run.stdout.strip().splitlines()]

    # ---- Python ground truth ----
    state_vars_py   = [x, y, z, alpha, beta, psi, x_dot, y_dot, z_dot,
                       alpha_dot, beta_dot, psi_dot]
    control_vars_py = [F1, T1, T2, T3]
    state_rhs_n = state_rhs.subs(params_subs)
    dyn_py = lambdify([t]+state_vars_py+control_vars_py, state_rhs_n, 'numpy')

    x_test = np.array([1.0, -2.0, 50.0, 0.05, -0.03, 0.0,
                       0.5, -0.1, -10.0, 0.01, -0.02, 0.0])
    u_test = np.array([100.0, 0.5, -0.3, 0.0])
    userF_test = np.array([2.5, -1.5, 0.7])
    m_val_local = 10.0

    dx_phys = np.array(dyn_py(0.0, *x_test, *u_test)).flatten()
    dx_phys[6] += userF_test[0] / m_val_local
    dx_phys[7] += userF_test[1] / m_val_local
    dx_phys[8] += userF_test[2] / m_val_local
    ref_pos_arr = np.array([0., 0., 0.])
    d_int = ref_pos_arr - x_test[:3]
    dx_full_py = np.concatenate([dx_phys, d_int])
    py_vals = list(dx_full_py)

    # ExecuteControl ground truth
    ax_v, ay_v, az_v = 0.5, -0.2, -3.0
    jx_v, jy_v, jz_v = 0.1, -0.05, 0.2
    sx_v, sy_v, sz_v = 0.01, -0.02, 0.03
    ff_subs = {ax_sym: ax_v, ay_sym: ay_v, az_sym: az_v,
               jx_sym: jx_v, jy_sym: jy_v, jz_sym: jz_v,
               sx_sym: sx_v, sy_sym: sy_v, sz_sym: sz_v}
    F1_ff_val = float(F1_ff_static.subs(ff_subs).subs(params_subs).evalf())
    T1_ff_val = float(T1_ff_static.subs(ff_subs).subs(params_subs).evalf())
    T2_ff_val = float(T2_ff_static.subs(ff_subs).subs(params_subs).evalf())

    x_aug_test = np.concatenate([x_test, [0.1, 0.2, -0.05]])
    u_lqr_py = -K_e @ x_aug_test
    u_total_py = np.array([F1_ff_val, T1_ff_val, T2_ff_val, 0.0]) + u_lqr_py
    F1_max_val = 700.0
    u_total_py[0] = max(0.0, min(F1_max_val, u_total_py[0]))
    py_vals += list(u_total_py)

    c_arr = np.array(c_vals); py_arr = np.array(py_vals)
    abs_err = np.abs(c_arr - py_arr)
    rel_err = abs_err / np.maximum(np.abs(py_arr), 1e-300)

    print(f"Total values compared: {len(c_arr)}")
    print(f"Max abs error C++ vs Python: {abs_err.max():.3e}")
    print(f"Max rel error C++ vs Python: {rel_err.max():.3e}")
    assert abs_err.max() < 1e-9, "C++ export disagrees with Python — investigate"
    print("\n*** C++ EXPORT VALIDATED ***")